# Anomaly Detection and Finetuning

The goal here is to train a Isolation Forest anomaly detection model and finetune a 3B-parameter LLM in order to detect satellite system abnormalities and deliver a report for mission ops teams.
The models will be trained using the [OPSSAT-AD](https://zenodo.org/records/12588359) dataset.

### Carve out the train, validation, and test pool from the OPSSAT-AD dataset:

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("https://raw.githubusercontent.com/Tegh25/Satellite-Anomaly-Detector/refs/heads/main/model/dataset.csv")

test_df = df[df["train"] == 0]
train_pool = df[df["train"] == 1]

# Carve validation out of the train pool only, stratify so the ~20% anomaly rate
# is preserved in both slices, not skewed by chance
train_df, val_df = train_test_split(
    train_pool, test_size=0.15, stratify=train_pool["anomaly"], random_state=42
)

print(len(train_df), len(val_df), len(test_df))

1354 240 529


### Anomaly Detection Model

Two models will be compared:
1. Isolation Forest (unsupervised)
2. Random Forest (supervised)

**Worth comparing against a supervised baseline.** Isolation Forest is unsupervised and it never uses the `anomaly` labels during training, only for threshold tuning afterward. Given this dataset has anomaly labels, a supervised model will likely outperform it directly. The unsupervised approach is still justified, like it can catch novel anomaly types a supervised model was never trained on.

In [3]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_curve, classification_report

feature_cols = [
    "mean", "var", "std", "kurtosis", "skew",
    "n_peaks", "smooth10_n_peaks", "smooth20_n_peaks",
    "diff_peaks", "diff2_peaks", "diff_var", "diff2_var",
    "gaps_squared", "var_div_duration", "var_div_len",
]

detector = IsolationForest(contamination=0.2, random_state=42)
detector.fit(train_df[feature_cols])

# Tune the decision threshold on validation instead of using the default
# the default threshold is derived purely from `contamination` at fit time,
# not from actually optimizing precision/recall on held-out data
val_scores = -detector.decision_function(val_df[feature_cols])
precision, recall, thresholds = precision_recall_curve(val_df["anomaly"], val_scores)
f1 = 2 * precision * recall / (precision + recall + 1e-12)
best_threshold = thresholds[f1[:-1].argmax()]

# Testing
test_scores = -detector.decision_function(test_df[feature_cols])
test_preds = (test_scores >= best_threshold).astype(int)
print("Isolation Forest (unsupervised):")
print(classification_report(test_df["anomaly"], test_preds))

Isolation Forest (unsupervised):
              precision    recall  f1-score   support

           0       0.87      0.49      0.62       416
           1       0.28      0.74      0.41       113

    accuracy                           0.54       529
   macro avg       0.58      0.61      0.52       529
weighted avg       0.75      0.54      0.58       529



In [4]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
clf.fit(train_df[feature_cols], train_df["anomaly"])

sup_preds = clf.predict(test_df[feature_cols])
print("Random Forest (supervised):")
print(classification_report(test_df["anomaly"], sup_preds))

Random Forest (supervised):
              precision    recall  f1-score   support

           0       0.96      0.99      0.97       416
           1       0.95      0.83      0.89       113

    accuracy                           0.95       529
   macro avg       0.95      0.91      0.93       529
weighted avg       0.95      0.95      0.95       529

